**🧩 1. Problem & Data Description**
Skin cancer is one of the most common cancers globally. Early detection significantly increases the chances of successful treatment. This project aims to develop a Convolutional Neural Network (CNN) that can classify skin lesions into categories (e.g., benign vs malignant) using image data.

**Dataset**: 
We use the [Skin Cancer MNIST: HAM10000](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) dataset from Kaggle. It contains 10,015 dermatoscopic images of skin lesions categorized into seven classes.

**Goal**: 
Train a CNN model to predict the correct lesion class from an input image.


📊 2. Exploratory Data Analysis (EDA)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load metadata
data_path = '/kaggle/input/skin-cancer-mnist-ham10000/'
df = pd.read_csv(os.path.join(data_path, 'HAM10000_metadata.csv'))

# Preview dataset
df.head()


In [ ]:
# Class distribution
plt.figure(figsize=(10, 5))
sns.countplot(x='dx', data=df, order=df['dx'].value_counts().index)
plt.title('Class Distribution')
plt.show()

# Map images to class names
class_map = {
    'akiec': 'Actinic Keratoses',
    'bcc': 'Basal Cell Carcinoma',
    'bkl': 'Benign Keratosis',
    'df': 'Dermatofibroma',
    'mel': 'Melanoma',
    'nv': 'Melanocytic Nevi',
    'vasc': 'Vascular Lesions'
}
df['class_name'] = df['dx'].map(class_map)


🧠 3. Data Preprocessing and Augmentation

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
import cv2

# Resize and load images
image_dir = os.path.join(data_path, 'HAM10000_images_part_1')
image_size = 64

# Combine images from part 1 and part 2
img_data = []
img_labels = []

for index, row in df.iterrows():
    img_id = row['image_id']
    for part in ['part_1', 'part_2']:
        path = os.path.join(data_path, f'HAM10000_images_{part}', f'{img_id}.jpg')
        if os.path.exists(path):
            img = cv2.imread(path)
            img = cv2.resize(img, (image_size, image_size))
            img_data.append(img)
            img_labels.append(row['dx'])
            break

X = np.array(img_data) / 255.0
y = pd.get_dummies(img_labels).values


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# Data augmentation
datagen = ImageDataGenerator(
    rotation_range=20,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True
)
datagen.fit(X_train)


🏗️ 4. Model Building (CNN)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization

model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(image_size, image_size, 3)),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(64, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Conv2D(128, (3, 3), activation='relu'),
    BatchNormalization(),
    MaxPooling2D(2, 2),
    
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(y.shape[1], activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()



🏋️‍♂️ 5. Training the Model

In [ ]:
history = model.fit(
    datagen.flow(X_train, y_train, batch_size=32),
    epochs=20,
    validation_data=(X_test, y_test)
)


📈 6. Results and Evaluation

In [ ]:
# Plot accuracy and loss
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validation')
plt.title('Model Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validation')
plt.title('Model Loss')
plt.legend()

plt.show()


In [ ]:
# Evaluate on test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test Accuracy: {test_acc:.2f}')


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import numpy as np

# Predict class probabilities
y_pred_probs = model.predict(X_test)

# Convert one-hot encoded vectors to class labels
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Generate class names from one-hot labels
class_names = pd.get_dummies(df['dx']).columns.tolist()

# Classification report
report = classification_report(y_true, y_pred, target_names=class_names, output_dict=False)
print("📋 Classification Report:\n")
print(report)


In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_true, y_pred)

# Plot it
plt.figure(figsize=(10, 7))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')
plt.show()
